In [27]:
from DeviceClasses.E5063A_VNA_Class import VNA
from DeviceClasses.RS_FSW_new import FSW
from DeviceClasses.SMA100B import Signal_Generator
from DeviceClasses.Utilities.Fitting import fit_S21_peak, S21_eq, linear_voltage_to_dB, dB_to_linear_voltage
from DeviceClasses.Utilities.data_management import read_file, save_data_3, attenuation_object

import plotly.graph_objects as go
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import os

%matplotlib qt



In [28]:
#base paths
main_dir = r".\\"
base_path = main_dir + r"circuit_2\\data\\"
S21_data_path = base_path + "\\raw_data\\"
print(os.listdir(main_dir))

rel_S21_data_path = "Averaged_S21.tsv"


['Attenuation_Processor_averager.ipynb', 'Attenuation_Processor_circuit_maker.ipynb', 'circuit_2', 'S21.ipynb', 'S_param_analysis.ipynb', 'zurich_mixer_attenuation.ipynb']


In [29]:
#data loader

#organisation of dictionary
# SA_data = {"10[dBm]":{
#     "average_21":{
#         "freq"          :{},
#         "time"          :{},
#         "IQ_voltage_mag":{},
#     }
# }}
data = {}
atten_dict = {}

#load in file
for folder in os.listdir(base_path):
    print(f"folder:{folder}")
    S21_data_path = base_path +f"\\{folder}\\" + rel_S21_data_path
    print(f"path:{S21_data_path}")
    data_pandas_temp, meta_data_temp = read_file(S21_data_path)
    atten = attenuation_object(data_pandas_temp["S_parameter[dB]"],data_pandas_temp["Frequency[Hz]"], data_pandas_temp["S_parameter_upper_std[dB]"], data_pandas_temp["S_parameter_lower_std[dB]"])
    atten_dict[folder] = atten
    data[folder]= {
        "atten":atten,
        "mag":data_pandas_temp["S_parameter[dB]"],
        "freq":data_pandas_temp["Frequency[Hz]"],
        "data":data_pandas_temp,
        "meta_data":meta_data_temp}
        


folder:2024_04_10_Inside_Fridge_input_line
path:.\\circuit_2\\data\\\2024_04_10_Inside_Fridge_input_line\Averaged_S21.tsv
folder:2024_04_10_inside_fridge_output_line
path:.\\circuit_2\\data\\\2024_04_10_inside_fridge_output_line\Averaged_S21.tsv


In [30]:

print(f"keys:")
for key in data.keys():
    print(f"{key}")



keys:
2024_04_10_Inside_Fridge_input_line
2024_04_10_inside_fridge_output_line


In [31]:
#need to save folders for:
plt.close()
# VNA_circuit
    # made up of both ends of the circuit, ff1, and ff12x20dBattenuators, fridge input and output,
# VNA_power_incident_on_cavity
    #just the first half of the circuit and ff1, fridge input
# SG_power_incident_on_cavity
    #SG circuit, fridge input, ff1, ff2
# SA_power_output_of_cavity
    #SA circuit, fridge output, ff12x20dBattenuators, ff2


#VNA
VNA_to_cavity_atten = atten_dict["2024_04_10_Inside_Fridge_input_line"]
VNA_cavity_to_VNA_atten = atten_dict["2024_04_10_inside_fridge_output_line"]
VNA_S21_extra_atten = VNA_to_cavity_atten + VNA_cavity_to_VNA_atten

atten_circuits = {
    "VNA_to_cavity_atten":VNA_to_cavity_atten,
    "VNA_cavity_to_VNA_atten":VNA_cavity_to_VNA_atten,
    "VNA_S21_extra_atten":VNA_S21_extra_atten,
} | atten_dict

fig_plotly_1 = go.Figure()
fig_plotly_1.update_layout(
    title="S-parameters",
    xaxis_title="Frequency [Hz]",
    yaxis_title="Power Ratio [dB]"
    )

for key, value in atten_circuits.items():
    print(key)
    file_name = f"atten_{key}.tsv"
    save_file_location = main_dir + r"\circuit_2\processed_data\\"
    headers = "Frequency[Hz]\tattenuation[dB]\tatten_upper_std[dB]\tatten_lower_std[dB]"
    meta_data_saved = {"circuit_name":key}
    save_data = np.array([value.frequencies, value.magnitude, value.upper_error, value.lower_error])
    save_data_3(save_data, file_name, save_file_location, headers, meta_data = meta_data_saved )

    if len(value.frequencies) == 1:
        fig_plotly_1.add_trace(go.Scatter(x=value.frequencies, 
                                          y=value.magnitude, 
                                          mode='markers', 
                                          name=f"{key}",
                                          error_y=dict(
                                                type='data',
                                                symmetric=False,
                                                array=value.upper_error,
                                                arrayminus=value.lower_error)))
        
    else:
        fig_plotly_1.add_trace(go.Scatter(x=value.frequencies, y=value.magnitude, mode='lines', name=f"{key}",legendgroup = f"{key}" ))
        fig_plotly_1.add_trace(go.Scatter(x=value.frequencies, y=value.magnitude+value.upper_error, legendgroup = f"{key}",  showlegend = False   ))
        fig_plotly_1.add_trace(go.Scatter(x=value.frequencies, y=value.magnitude-value.lower_error, fill="tonexty", legendgroup = f"{key}", showlegend = False ))



    file_name = f"atten_circuit_2_all_PLOTLY_plot.html"
    save_path = f"{save_file_location}\\" + file_name


fig_plotly_1.write_html(save_path)




VNA_to_cavity_atten
datashape:(4, 10001)
VNA_cavity_to_VNA_atten
datashape:(4, 10001)
VNA_S21_extra_atten
datashape:(4, 10001)
2024_04_10_Inside_Fridge_input_line
datashape:(4, 10001)
2024_04_10_inside_fridge_output_line
datashape:(4, 10001)


In [32]:
value = VNA_to_cavity_atten
#value = VNA_cavity_to_VNA_atten
#value = VNA_S21_extra_atten
#value = atten_dict["2025_04_14_Fridge_circuit_analysis_input_line"]
#value = _2_20_dB_attenuators
value = atten_dict["2025_01_22_circuit2_VnaCircuitInputToFridgeInputFf1"]
#value = atten_dict["2025_04_14_Fridge_circuit_analysis_input_line"] 
#value = atten_dict["2025_01_22_circuit2_VnaCircuitInputToFridgeInputFf1"]
#value = atten_dict["2025_01_22_circuit2_VnaCircuitInputToFridgeInputFf1"]+ atten_dict["2025_04_14_Fridge_circuit_analysis_input_line"]


print(dB_to_linear_voltage(value.magnitude))


save_data = [value.frequencies, value.magnitude, value.upper_error, value.lower_error]
for i in save_data:
    print(np.shape(i))

KeyError: '2025_01_22_circuit2_VnaCircuitInputToFridgeInputFf1'

In [ ]:
print(0.0009795673494340062)


value_linear, upper_error, lower_error= [0.00095513,
0.0009795673494340062,
0.0009795673494340062]

upper_value_linear = value_linear + upper_error
lower_value_linear = value_linear - lower_error
upper_value_dB = linear_voltage_to_dB(upper_value_linear)
lower_value_dB = linear_voltage_to_dB(lower_value_linear)
value_dB = linear_voltage_to_dB(value_linear)
upper_error_dB = upper_value_dB - value_dB
lower_error_dB = np.abs(lower_value_dB - value_dB)

print(f"{lower_value_linear=}")
print(f"{lower_error}")
print(f"{lower_value_dB=}")
print(value_dB, upper_error_dB, lower_error_dB)




0.0009795673494340062
lower_value_linear=-2.4437349434006236e-05
0.0009795673494340062
lower_value_dB=nan
-60.39875027638192 6.131011009723856 nan


c:\pythonscripts\device_classes\DeviceClasses\Utilities\Fitting.py:89: RuntimeWarning:

invalid value encountered in log10



In [ ]:
#attenuation circuit 2:
VNA_to_cavity_atten = attenuation_object.load_from_tsv(main_dir + r"\Attenuation\circuit_2\processed_data\atten_VNA_to_cavity_atten.tsv")
VNA_cavity_to_VNA_atten = attenuation_object.load_from_tsv(main_dir + r"\Attenuation\circuit_2\processed_data\atten_VNA_cavity_to_VNA_atten.tsv")
VNA_S21_extra_atten = attenuation_object.load_from_tsv(main_dir + r"\Attenuation\circuit_2\processed_data\atten_VNA_S21_extra_atten.tsv")
SG_to_cavity_atten = attenuation_object.load_from_tsv(main_dir + r"\Attenuation\circuit_2\processed_data\atten_SG_to_cavity_atten.tsv")
SA_cavity_to_SA_attenuation = attenuation_object.load_from_tsv(main_dir + r"\Attenuation\circuit_2\processed_data\atten_SA_cavity_to_SA_attenuation.tsv")

In [ ]:
print(SA_cavity_to_SA_attenuation.lower_error)

[4.56710123 4.56727859 4.56724864 ... 4.56710862 4.56709366 4.56708851]
